# NeuralNetwork.ipynb

Purpose: Create a neural network to model launch scrub/no scrub based on weather conditions.


In [ ]:
import pandas as pd
import numpy as np
import pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F


### Data Gathering

Data was produced from Bravo-Wx-Launch hackathon team. They created the raw-data-transform-multi.py script to compile of the data. I took all of this data and placed it into a single dataframe for analysis.

In [ ]:
if pathlib.Path('combine.csv').exists():
    df = pd.read_csv('combine.csv',index_col=0,header=0)
else:
    base_dir = pathlib.Path('/home/bearmint/projects/bravo-wx-launch/bravo-wx-launch/test-runs/test-run-20220722-0845 (complete)')
    df_list = []
    for filename in list(pathlib.Path.iterdir(base_dir)):
        with open(pathlib.Path.joinpath(base_dir, filename), 'r') as f:
            df_list.append(pd.read_csv(f))
            
    df = pd.concat(df_list, ignore_index=True)
    df.set_index('Unnamed: 0', inplace=True)
    df.to_csv('combine.csv')

In [ ]:
df

In [ ]:
pd.options.display.max_columns = 200
df.scrub_id.value_counts()

In [ ]:
df.info(verbose=True, show_counts=True)

# Data Preprocessing
Quality issues
- Columns with NaNs: Dropping columns with more than 1000 NaNs. Then imputing column values with a KNN imputer using the mean from the 5 nearest values
  - Could potentially check each column to see if mean/mode/constant would be better to impute 

In [ ]:
#cols_to_remove = []
#for column in df.columns:
#    x = df[column].isna().sum()
#    if x >= 4000:
#        cols_to_remove.append(column)
        #print(f'{column}: {x}')

In [ ]:
#separate targets
target_df = df['scrub_id']
df.drop(['scrub_id'], axis=1, inplace=True)
#create df with few NaNs
df_low_nan = df.copy()
#find columns where NaNs are greater than 1000
cols_to_remove = [column for column in df_low_nan.columns if df_low_nan[column].isna().sum()>=1000]
df_low_nan.drop(cols_to_remove, axis=1, inplace=True)


In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

def make_data(df, target_df, imputer):
    #splits data, imputes missing values, normalizes it and then returns data ready to be put in data laoders
    X_train, X_test, y_train, y_test = train_test_split(df, target_df,random_state=42)

    pipe = Pipeline([('imputer', imputer), ('normalizer', Normalizer())])
    train_df = pd.DataFrame(data=pipe.fit_transform(X_train),columns=df.columns)
    train_data = []
    for i in range(len(train_df.to_numpy())):
        train_data.append([train_df.to_numpy()[i], y_train.to_numpy()[i]])

    test_df = pd.DataFrame(data=pipe.fit_transform(X_test), columns=df.columns)
    test_data = []
    for i in range(len(test_df.to_numpy())):
        test_data.append([test_df.to_numpy()[i], y_test.to_numpy()[i]])
    
    return train_data, test_data



In [ ]:
train_data, test_data = make_data(df_low_nan, target_df, KNNImputer(n_neighbors=5))
#train_data

# Data Loaders

In [ ]:
def create_loaders(train_data, test_data, batch_size, valid_size):
    """
    Takes in training and test data and creates DataLoader objects
    Inputs:
        train_data: Transformed training data (must be torchvision.datasets.folder object (potentially only an ImageFolder))
        test_data: Transformed testing data (must be torchvision.datasets.folder object (potentially only an ImageFolder))
        batch_size: How many samples per batch (int)
        valid_size: Percentage of training set to use for validation (float)
    Outputs:
        train_loader: Training data loader (DataLoader object)
        valid_loader: Validation data loader (DataLoader object)
        test_loader: Testing data loader (DataLoader object)
    """

    #obtain indices for validation
    num_train = len(train_data)
    indices = list(range(num_train))
    np.random.shuffle(indices)
    split = int(np.floor(valid_size*num_train))
    train_idx, valid_idx = indices[split:], indices[:split]

    #samplers to select train/valid batches
    train_sampler = torch.utils.data.sampler.SubsetRandomSampler(train_idx)
    valid_sampler = torch.utils.data.sampler.SubsetRandomSampler(valid_idx)

    #create data loaders by combining data and samplers
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=train_sampler)
    valid_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=valid_sampler)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size)

    return train_loader, valid_loader, test_loader

train_loader, valid_loader, test_loader = create_loaders(train_data, test_data, batch_size=16, valid_size=0.2)
loaders = {'train': train_loader, 'valid': valid_loader, 'test': test_loader}

In [ ]:
#Used to check data format and that each input is an observation and its corresponding label - uncomment if you want to see the data
#train_iter = iter(train_loader)
#data, label = train_iter.next()
#print(data.float(),label)
#print(len(data[0]))

### Initialize cuda 

In [ ]:
use_cuda = torch.cuda.is_available()
if use_cuda:
    print('Train on GPU')
else:
    print("Train on CPU")

### Build Network

In [ ]:
class Network(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.fc1 = nn.Linear(30,20)
        self.fc2 = nn.Linear(20,10)
        self.fc3 = nn.Linear(10,2)

        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(x.shape[0], -1)

        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)

        return x
model_low_nan = Network()
model_low_nan



### Loss Function and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
def get_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=0.001)

### Train Model

In [ ]:
def train(n_epochs, loaders, model, optimizer, criterion, use_cuda, save_path):
    """returns trained model"""
    # initialize tracker for minimum validation loss
    valid_loss_min = np.Inf 
    
    model.cuda()
    
    for epoch in range(1, n_epochs+1):
        # initialize variables to monitor training and validation loss
        train_loss = 0.0
        valid_loss = 0.0
        
        ###################
        # train the model #
        ###################
        # set the module to training mode
        model.train()
        for data, target in loaders['train']:
            # move to GPU
            if use_cuda:
                data, target = data.cuda(), target.cuda()
            data = data.float()
            #clear gradients
            optimizer.zero_grad()
            #forward pass
            output = model(data)
            #calculate loss for batch
            loss = criterion(output, target)
            #backward pass
            loss.backward()
            #take optimization step
            optimizer.step()
            #update training loss
            train_loss += loss.item()*data.size(0)    

        ######################    
        # validate the model #
        ######################
        # set the model to evaluation mode
        model.eval()
        for batch_idx, (data, target) in enumerate(loaders['valid']):
            # move to GPU
            if use_cuda:
                data, target = data.cuda(), target.cuda()
            data = data.float()
            #forward pass
            output = model(data)
            #calculate loss
            loss = criterion(output, target)
            #update validation loss
            valid_loss += loss.item()*data.size(0)

        train_loss = train_loss/len(loaders['train'].sampler)
        valid_loss = valid_loss/len(loaders['valid'].sampler)

        # print training/validation statistics 
        print('Epoch: {} \tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(
            epoch, 
            train_loss,
            valid_loss
            ))

        if valid_loss <= valid_loss_min:
            print(f'Validation loss decreased ({valid_loss_min:.6f} --> {valid_loss:.6f}. Saving model...')
            torch.save(model.state_dict(), save_path)
            valid_loss_min = valid_loss
        

        
        
    return model


In [ ]:
model_low_nan = train(200, loaders, model_low_nan, get_optimizer(model_low_nan), criterion, use_cuda, 'model_low_nan.pt')

### Test Model

In [ ]:
def test(loaders, model, criterion, use_cuda):

    # monitor test loss and accuracy
    test_loss = 0.
    correct = 0.
    total = 0.

    # set the module to evaluation mode
    model.eval()

    for batch_idx, (data, target) in enumerate(loaders['test']):
        # move to GPU
        if use_cuda:
            data, target = data.cuda(), target.cuda()
        data = data.float()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the loss
        loss = criterion(output, target)
        # update average test loss 
        test_loss = test_loss + ((1 / (batch_idx + 1)) * (loss.data.item() - test_loss))
        # convert output probabilities to predicted class
        pred = output.data.max(1, keepdim=True)[1]
        # compare predictions to true label
        correct += np.sum(np.squeeze(pred.eq(target.data.view_as(pred))).cpu().numpy())
        total += data.size(0)
            
    print('Test Loss: {:.6f}\n'.format(test_loss))

    print('\nTest Accuracy: %2d%% (%2d/%2d)' % (
        100. * correct / total, correct, total))

# load the model that got the best validation accuracy
model_low_nan.load_state_dict(torch.load('model_low_nan.pt'))
test(loaders, model_low_nan, criterion, use_cuda)

### Model with imputed NaNs

Since there are so many missing NaNs three different imputing approaches will be taken
1. Median Imputation
2. KNN Imputation
3. Iterative Imputation

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

train_data, test_data = make_data(df, target_df, SimpleImputer(strategy='median'))
train_loader, valid_loader, test_loader = create_loaders(train_data, test_data, batch_size=16, valid_size=0.2)
loaders = {'train': train_loader, 'valid': valid_loader, 'test': test_loader}

In [ ]:
class Network2(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(174,130)
        self.fc2 = nn.Linear(130,90)
        self.fc3 = nn.Linear(90,50)
        self.fc4 = nn.Linear(50,2)

        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(x.shape[0], -1)

        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)

        return x
model_median = Network2()
model_KNN = Network2()
model_iter = Network2()

In [ ]:
model_median = train(200, loaders, model_median, get_optimizer(model_median), criterion, use_cuda, 'model_median.pt')

In [ ]:
model_median.load_state_dict(torch.load('model_median.pt'))
test(loaders, model_median, criterion, use_cuda)

### KNN Model

In [ ]:
train_data, test_data = make_data(df, target_df, KNNImputer(n_neighbors=5))
train_loader, valid_loader, test_loader = create_loaders(train_data, test_data, batch_size=16, valid_size=0.2)
loaders = {'train': train_loader, 'valid': valid_loader, 'test': test_loader}
model_KNN = train(200, loaders, model_KNN, get_optimizer(model_KNN), criterion, use_cuda, 'model_KNN.pt')

In [ ]:
model_KNN.load_state_dict(torch.load('model_KNN.pt'))
test(loaders, model_KNN, criterion, use_cuda)

### Iter Model

In [ ]:
train_data, test_data = make_data(df, target_df, IterativeImputer())
train_loader, valid_loader, test_loader = create_loaders(train_data, test_data, batch_size=16, valid_size=0.2)
loaders = {'train': train_loader, 'valid': valid_loader, 'test': test_loader}
model_iter = train(200, loaders, model_iter, get_optimizer(model_iter), criterion, use_cuda, 'model_iter.pt')

In [ ]:
model_iter.load_state_dict(torch.load('model_iter.pt'))
test(loaders, model_iter, criterion, use_cuda)

### Findings
Most models have similar accuracy on the imputed data. The median imputation takes the shortest and iterative imputation takes the longest.

More data would greatly aid the first model as the accuracy from the first model was low compared to the last three models (since the first model only had 30 features to train on). Even though the last 3 models had more accuracy, it may still be better to utilize the first model since extensive data was imputed on the last 3 and may not hold up to more real world data.